# Notebook: 03 Training
### Purpose: create train and validation datasets, build augmentations, run Trainer, save versioned checkpoints.


In [1]:
import os
import sys


sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))

import random

import torch

from src.data.annotations import load_json_annotations
from src.data.augmentations import get_train_augmentations, get_val_augmentations
from src.data.loaders import ImageMaskDataset
from src.models.zoo import MODEL_BUILDERS
from src.training.engine import run_training
from src.utils.config import Config
from src.utils.helpers import init_notebook, p


config = Config.load()

init_notebook(config.train.seed)

train_dir = config.paths.train_images
annotations_path = config.paths.annotations
entries = load_json_annotations(annotations_path)

# Shuffle entries
random.shuffle(entries)


None: None
=== init_notebook ===
Done


#### Dataset split

In [2]:
# Compute number of validation samples (20 percent of dataset)
val_count = max(1, int(0.2 * len(entries)))

# Split validation set, and training set
val_entries = entries[:val_count]
train_entries = entries[val_count:]

# Build augmentation pipelines for training and validation
train_tf = get_train_augmentations(config.train.image_size)
val_tf = get_val_augmentations(config.train.image_size)

# Build dataset objects that load image-mask pairs and apply transforms
train_ds = ImageMaskDataset(train_entries, train_dir, transform = train_tf)
val_ds = ImageMaskDataset(val_entries, train_dir, transform = val_tf)

# Build dataloaders
train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size = config.train.batch_size,
        shuffle = True,
        num_workers = config.train.num_workers,
)

val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size = config.train.batch_size,
        shuffle = False,
        num_workers = config.train.num_workers,
)

p("Train samples", len(train_ds))
p("Val samples", len(val_ds))

Train samples: 120
Val samples: 30


#### Available Models

In [3]:
p("Models", MODEL_BUILDERS)
#config.show()
p("Batch", config.train.batch_size)
p("Epochs", config.train.epochs)
p("Learning Rate", config.train.learning_rate, precision = 9)
p("Image Size", config.train.image_size)


Models: 8 keys
  simple_cnn: <function create_simple_cnn at 0x000001F9FE754040>
  unet: <function create_unet at 0x000001F9FF808900>
  smp_unet: <function create_smp_unet at 0x000001F9FF808400>
  smp_fpn: <function create_smp_fpn at 0x000001F9FF808C20>
  smp_linknet: <function create_smp_linknet at 0x000001F9FF808CC0>
  smp_deeplabv3: <function create_smp_deeplabv3 at 0x000001F9FF808D60>
  smp_deeplabv3plus: <function create_smp_deeplabv3plus at 0x000001F9FF808E00>
  segformer: <function create_segformer at 0x000001F9FF808EA0>
Batch: 8
Epochs: 10
Learning Rate: 0.000100000
Image Size: 512


#### Run Training

In [4]:
version_root = config.paths.models
trainer = run_training(
        config = config,
        train_loader = train_loader,
        val_loader = val_loader,
        version_root = version_root,
        model_name = "simple_cnn",
)

=== Training started ===
=== SimpleCNN(
  (encoder): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (2): Sequential(
      (0): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (head): Conv2d(32, 1, kernel_size=(1, 1), stride=(1, 1))
) ===
[Warn]: Epoch 1
[Info]: Train loss 0.6847
[Info]: Val loss 0.6422
[Info]: IoU 0.3665
[Info]: Dice 0.5254
[Info]: Acc 0.6995
[Info]: New best model
[Warn]: Epoch 2
[Info]: Train loss 0.6337
[In